In [ ]:
##### Copyright 2025 Google LLC.

# 🎓 CampusGenius – Multi-Agent Study Companion (Capstone Project)

This notebook implements **CampusGenius**, a multi-agent study companion for students.

It uses:

- **NotesMaster Agent** – a remote A2A agent for notes, explanations, and flashcards  
- **CampusGenius Assistant** – a user-facing agent that plans study schedules and delegates work to NotesMaster  

You’ll see:

- How to design a realistic, student-focused agent system  
- How to expose an agent via **A2A**  
- How another agent can consume it using `RemoteA2aAgent`  

This is my submission for the **Agents Intensive – Capstone Project**.


In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# You may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.


## 📚 Getting Started (Kaggle)

1. Click **Copy & Edit** to make your own copy.  
2. Go to **Add-ons → Secrets** and create a secret called `GOOGLE_API_KEY`.  
3. Paste your Gemini API key there and attach it to this notebook.  
4. Run cells from top to bottom (don’t use Run All to avoid rate limits).


In [2]:
!pip install -q google-adk --no-deps

In [2]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ API key loaded successfully.")
except Exception as e:
    print("❌ Please add GOOGLE_API_KEY under Add-ons → Secrets.")
    print("Error:", e)

✅ API key loaded successfully.


In [3]:
import json
import requests
import subprocess
import time
import uuid
import warnings

from google.adk.agents import LlmAgent
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent, AGENT_CARD_WELL_KNOWN_PATH
from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

warnings.filterwarnings("ignore")

print("✅ ADK imported successfully.")

retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],
)

print("⏳ Retry policy configured.")


✅ ADK imported successfully.
⏳ Retry policy configured.


In [4]:
def generate_notes(topic: str, extra: str = "") -> str:
    """
    Tool: Prepare structured exam notes.
    The LLM will turn this structure into real notes.
    """
    text = (
        f"Topic: {topic}\n\n"
        "Create structured notes with:\n"
        "- Short overview\n"
        "- Key definitions\n"
        "- Important points\n"
        "- Example(s)\n"
        "- Quick revision bullets\n"
    )
    if extra:
        text += f"\nExtra student context:\n{extra}\n"
    return text


def explain_topic(topic: str, level: str = "beginner") -> str:
    """
    Tool: Ask for a simple explanation.
    """
    return (
        f"Explain '{topic}' to a {level} level student "
        "using simple language, clear steps, and at least one real-world example. "
        "End with a one-line summary."
    )


def create_flashcards(topic: str) -> str:
    """
    Tool: Ask for flashcards.
    """
    return (
        f"Generate 8 numbered Q/A flashcards for '{topic}'. "
        "Format strictly as:\nQ: ...\nA: ...\nKeep them exam-focused and concise."
    )


notes_master_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="notes_master_agent",
    description="Academic specialist for notes, explanations, and flashcards.",
    instruction=(
        "You are NotesMaster, a specialist academic agent helping college students.\n\n"
        "For generate_notes:\n"
        "- Turn the structure into clean, formatted notes with headings and bullets.\n\n"
        "For explain_topic:\n"
        "- Give a simple explanation with examples and a final one-line summary.\n\n"
        "For create_flashcards:\n"
        "- Produce numbered flashcards in 'Q: ... / A: ...' format.\n\n"
        "Always keep the language student-friendly and exam-oriented."
    ),
    tools=[generate_notes, explain_topic, create_flashcards],
)

print("✅ NotesMaster Agent created.")


✅ NotesMaster Agent created.


In [5]:
notes_master_server_code = '''
import os
from google.adk.agents import LlmAgent
from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.adk.models.google_llm import Gemini
from google.genai import types

retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],
)

def generate_notes(topic: str, extra: str = "") -> str:
    text = (
        f"Topic: {topic}\\n\\n"
        "Create structured notes with:\\n"
        "- Short overview\\n"
        "- Key definitions\\n"
        "- Important points\\n"
        "- Example(s)\\n"
        "- Quick revision bullets\\n"
    )
    if extra:
        text += f"\\nExtra student context:\\n{extra}\\n"
    return text

def explain_topic(topic: str, level: str = "beginner") -> str:
    return (
        f"Explain '{topic}' to a {level} level student "
        "using simple language, clear steps, and at least one real-world example. "
        "End with a one-line summary."
    )

def create_flashcards(topic: str) -> str:
    return (
        f"Generate 8 numbered Q/A flashcards for '{topic}'. "
        "Format strictly as:\\nQ: ...\\nA: ...\\nKeep them exam-focused and concise."
    )

notes_master_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="notes_master_agent",
    description="Academic specialist for notes, explanations, and flashcards.",
    instruction=(
        "You are NotesMaster, a specialist academic agent helping college students."
    ),
    tools=[generate_notes, explain_topic, create_flashcards],
)

app = to_a2a(notes_master_agent, port=8001)
'''

with open("/tmp/notes_master_server.py", "w") as f:
    f.write(notes_master_server_code)

print("📝 NotesMaster server code written to /tmp/notes_master_server.py")

notes_master_process = subprocess.Popen(
    [
        "uvicorn",
        "notes_master_server:app",
        "--host", "localhost",
        "--port", "8001",
    ],
    cwd="/tmp",
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env={**os.environ},
)

print("🚀 Starting NotesMaster A2A server on http://localhost:8001 ...")

for i in range(30):
    try:
        r = requests.get("http://localhost:8001/.well-known/agent-card.json", timeout=2)
        if r.status_code == 200:
            print("✅ NotesMaster server is running!")
            break
    except Exception:
        time.sleep(2)
        print(".", end="", flush=True)
else:
    print("\n⚠️ Could not confirm NotesMaster server startup. Check logs if needed.")


📝 NotesMaster server code written to /tmp/notes_master_server.py
🚀 Starting NotesMaster A2A server on http://localhost:8001 ...
...........✅ NotesMaster server is running!


In [6]:
try:
    r = requests.get("http://localhost:8001/.well-known/agent-card.json", timeout=5)
    print("📋 NotesMaster Agent Card:")
    print(json.dumps(r.json(), indent=2))
except Exception as e:
    print("❌ Failed to fetch agent card:", e)


📋 NotesMaster Agent Card:
{
  "capabilities": {},
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "description": "Academic specialist for notes, explanations, and flashcards.",
  "name": "notes_master_agent",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "skills": [
    {
      "description": "Academic specialist for notes, explanations, and flashcards. I am NotesMaster, a specialist academic agent helping college students.",
      "id": "notes_master_agent",
      "name": "model",
      "tags": [
        "llm"
      ]
    },
    {
      "description": "Call self as a function.",
      "id": "notes_master_agent-generate_notes",
      "name": "generate_notes",
      "tags": [
        "llm",
        "tools"
      ]
    },
    {
      "description": "Call self as a function.",
      "id": "notes_master_agent-explain_topic",
      "name": "explain_topic",
      "tags": [
        "llm",
        "tools"
      ]
    },
    

In [7]:
remote_notes_agent = RemoteA2aAgent(
    name="notes_master_agent",
    description="Remote NotesMaster academic agent (A2A).",
    agent_card=f"http://localhost:8001{AGENT_CARD_WELL_KNOWN_PATH}",
)

print("✅ Remote NotesMaster proxy created.")

campus_genius = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="campus_genius",
    description="CampusGenius – main multi-capability study companion.",
    instruction=(
        "You are CampusGenius, a study assistant for college students.\n\n"
        "You have access to a powerful academic sub-agent called NotesMaster.\n\n"
        "Rules:\n"
        "1. If the student asks for NOTES, SUMMARY, POINTS, or REVISION → "
        "call NotesMaster's generate_notes tool.\n"
        "2. If the student asks to EXPLAIN or UNDERSTAND a topic → "
        "call NotesMaster's explain_topic tool.\n"
        "3. If the student asks for FLASHCARDS, Q/A, REVISION CARDS → "
        "call NotesMaster's create_flashcards tool.\n"
        "4. If the student asks for a STUDY PLAN or SCHEDULE → "
        "create a realistic day-wise schedule yourself.\n"
        "5. Never ask for permission like 'Should I proceed?'. "
        "Always directly take the correct action.\n\n"
        "Always reply with clean formatting, headings, and bullet points."
    ),
    sub_agents=[remote_notes_agent],
)

print("✅ CampusGenius Agent created with NotesMaster as sub-agent.")


✅ Remote NotesMaster proxy created.
✅ CampusGenius Agent created with NotesMaster as sub-agent.


In [8]:
async def run_campus_genius(query: str):
    """
    Run a single interaction with CampusGenius.
    """
    session_service = InMemorySessionService()
    app_name = "campus_genius_app"
    user_id = "student_user"
    session_id = f"session_{uuid.uuid4().hex[:8]}"

    # Important: use keyword arguments for create_session in new ADK
    await session_service.create_session(
        app_name=app_name,
        user_id=user_id,
        session_id=session_id,
    )

    runner = Runner(
        agent=campus_genius,
        app_name=app_name,
        session_service=session_service,
    )

    content = types.Content(parts=[types.Part(text=query)])

    print(f"\n👤 Student: {query}\n")
    print("🧠 CampusGenius:\n" + "-" * 60)

    async for event in runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=content,
    ):
        if event.is_final_response() and event.content:
            for part in event.content.parts:
                if hasattr(part, "text"):
                    print(part.text)

    print("-" * 60)


In [9]:
import nest_asyncio, asyncio
nest_asyncio.apply()


In [10]:
await run_campus_genius("...")



👤 Student: ...

🧠 CampusGenius:
------------------------------------------------------------
What topic would you like to focus on today?
------------------------------------------------------------


In [11]:
await run_campus_genius(
    "Please give me exam-focused notes on Operating System Deadlocks in simple bullet points."
)



👤 Student: Please give me exam-focused notes on Operating System Deadlocks in simple bullet points.

🧠 CampusGenius:
------------------------------------------------------------


# Operating System Deadlocks: Exam-Focused Notes

## Overview

A deadlock in an operating system is a situation where two or more processes are unable to proceed because each is waiting for the other to release a resource. It's a common concurrency problem that can lead to system unresponsiveness.

## Key Definitions

*   **Process:** A program in execution.
*   **Resource:** Any hardware or software component that a process needs to complete its task (e.g., CPU, memory, I/O devices, files).
*   **Deadlock:** A state in which a set of processes are blocked because each process is holding a resource and waiting for another resource acquired by some other process in the same set.
*   **Mutual Exclusion:** At least one resource must be held in a non-sharable mode. Only one process can use the resource at any given time.
*   **Hold and Wait:** A process holds at least one resource and is waiting to acquire additional resources that are currently held by other processes.
*   **No Preemption

In [12]:
await run_campus_genius(
    "I am a BCA student. Explain normalization in DBMS in very simple words with an example."
)



👤 Student: I am a BCA student. Explain normalization in DBMS in very simple words with an example.

🧠 CampusGenius:
------------------------------------------------------------


Normalization in DBMS is a process used to organize data in a database efficiently. Its main goals are to eliminate redundant data and ensure data dependencies are logical. This makes the database more streamlined, easier to update, and less prone to errors.

Here's a breakdown in simple terms:

**What's the Problem Normalization Solves?**

Imagine you have a table storing student information, including their courses and instructor details. Without normalization, you might have the same instructor's details repeated for every student they teach. This leads to:

*   **Redundancy:** Wasting storage space by repeating information.
*   **Update Anomalies:** If an instructor's phone number changes, you'd have to update it in multiple places. Missing even one update can lead to inconsistent data.
*   **Deletion Anomalies:** If you delete a student, you might accidentally delete information about an instructor who only taught that one student.

**The Solution: Normalization**

Normalization b

In [13]:
await run_campus_genius(
    "Create 8 Q/A flashcards to revise the TCP/IP model."
)


👤 Student: Create 8 Q/A flashcards to revise the TCP/IP model.

🧠 CampusGenius:
------------------------------------------------------------


Q: What is the primary function of the Application Layer in the TCP/IP model?
A: To provide network services directly to user applications.

Q: Which layer in the TCP/IP model is responsible for logical addressing and routing?
A: The Internet Layer (or Network Layer).

Q: What is the main role of the Transport Layer in the TCP/IP model?
A: To provide reliable or unreliable data delivery services between hosts.

Q: Name the two main protocols operating at the Transport Layer of the TCP/IP model.
A: TCP (Transmission Control Protocol) and UDP (User Datagram Protocol).

Q: What does the Network Interface Layer (or Link Layer) handle in the TCP/IP model?
A: It deals with the physical transmission of data over the network medium and defines how data is presented to the network.

Q: What is the main difference between TCP and UDP?
A: TCP is connection-oriented and provides reliable, ordered delivery, while UDP is connectionless and offers faster, but less reliable, delivery.

Q: How does dat

In [14]:
await run_campus_genius(
    "My BCA 3rd semester exam is in 10 days and I can study 3 hours per day. Make a realistic day-wise study plan."
)


👤 Student: My BCA 3rd semester exam is in 10 days and I can study 3 hours per day. Make a realistic day-wise study plan.

🧠 CampusGenius:
------------------------------------------------------------
Here is a realistic 10-day study plan for your BCA 3rd-semester exams, assuming you can study 3 hours per day. This plan aims to cover your syllabus effectively while incorporating breaks and revision.

**Important Notes:**

*   **Subject Allocation:** This plan assumes you have 3-4 core subjects. Adjust the time spent on each subject based on its difficulty and your current understanding.
*   **Prioritization:** If some subjects are more heavily weighted or you find them more challenging, allocate more time to them.
*   **Active Recall:** Don't just passively read. Use active recall techniques like flashcards, practice questions, and explaining concepts to yourself.
*   **Breaks:** Take short breaks (5-10 minutes) every hour to maintain focus.
*   **Sleep & Health:** Ensure you get adequa